In [ ]:
## Please run after the model.py notebook to generate all the visualizations for the final report. This script will read the CSV outputs from model.py, create the necessary plots, and save them in high resolution for download. ##

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import confusion_matrix
from google.colab import files
import warnings

warnings.filterwarnings('ignore')

# Set standard plotting style for professional engineering reports
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("paper", font_scale=1.2)

generated_images = []

def save_plot(fig, filename):
    """Helper to save figures in high resolution and track them for download."""
    path = Path("output_figures") / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=300, bbox_inches='tight')
    generated_images.append(path)
    plt.close(fig)
    print(f"Saved figure: {filename}")

# =====================================================================
# PLOTTING FUNCTIONS
# =====================================================================

def create_model_comparison_plots(metrics_df, metric_col, title, filename):
    """Creates a bar chart comparing models based on a specific metric."""
    fig, ax = plt.subplots(figsize=(8, 5))

    # Sort for better visual flow
    is_ascending = True if metric_col in ['rmse', 'mae'] else False
    plot_df = metrics_df.sort_values(metric_col, ascending=is_ascending)

    sns.barplot(x=metric_col, y='model', data=plot_df, palette='viridis', ax=ax)

    ax.set_title(title, fontweight='bold')
    ax.set_xlabel(metric_col.upper())
    ax.set_ylabel('Model')

    # Add data labels
    for p in ax.patches:
        ax.annotate(f"{p.get_width():.3f}",
                    (p.get_width(), p.get_y() + p.get_height() / 2.),
                    ha='left', va='center', xytext=(5, 0), textcoords='offset points')

    save_plot(fig, filename)

def create_parity_plot(predictions_df, target_col, title, filename):
    """Creates an Actual vs. Predicted scatter plot with a line of perfect agreement."""
    fig, ax = plt.subplots(figsize=(6, 6))

    y_true = predictions_df[f"observed_{target_col}"]
    y_pred = predictions_df[f"predicted_{target_col}"]

    ax.scatter(y_true, y_pred, alpha=0.7, edgecolors='k')

    # Line of perfect agreement
    limits = [
        np.min([ax.get_xlim(), ax.get_ylim()]),
        np.max([ax.get_xlim(), ax.get_ylim()]),
    ]
    ax.plot(limits, limits, 'r--', zorder=0, label='Perfect Prediction')

    ax.set_aspect('equal')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel("Observed Pier Depth (ft)")
    ax.set_ylabel("Predicted Pier Depth (ft)")
    ax.legend()

    save_plot(fig, filename)

def create_residual_plot(predictions_df, target_col, title, filename):
    """Creates a residual plot to check for systematic error."""
    fig, ax = plt.subplots(figsize=(8, 5))

    y_true = predictions_df[f"observed_{target_col}"]
    y_pred = predictions_df[f"predicted_{target_col}"]
    residuals = y_true - y_pred

    ax.scatter(y_pred, residuals, alpha=0.7, edgecolors='k')
    ax.axhline(0, color='r', linestyle='--')

    ax.set_title(title, fontweight='bold')
    ax.set_xlabel("Predicted Pier Depth (ft)")
    ax.set_ylabel("Residuals (Observed - Predicted) [ft]")

    save_plot(fig, filename)

def create_feature_importance_plot(importance_df, title, filename):
    """Creates a horizontal bar chart of feature importances."""
    fig, ax = plt.subplots(figsize=(10, 6))

    # Take top 10 features if there are many one-hot encoded variables
    plot_df = importance_df.head(10)

    sns.barplot(x='importance', y='feature', data=plot_df, palette='magma', ax=ax)

    ax.set_title(title, fontweight='bold')
    ax.set_xlabel("Relative Importance")
    ax.set_ylabel("Structural Feature")

    save_plot(fig, filename)

def create_confusion_matrix_plot(predictions_df, target_col, title, filename):
    """Creates a heatmap confusion matrix for classification results."""
    fig, ax = plt.subplots(figsize=(6, 5))

    y_true = predictions_df[f"observed_{target_col}"]
    y_pred = predictions_df[f"predicted_{target_col}"]

    cm = confusion_matrix(y_true, y_pred)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Shallow (0)', 'Deep (1)'],
                yticklabels=['Shallow (0)', 'Deep (1)'])

    ax.set_title(title, fontweight='bold')
    ax.set_xlabel("Predicted Foundation Type")
    ax.set_ylabel("Observed Foundation Type")

    save_plot(fig, filename)


# =====================================================================
# MAIN EXECUTION
# =====================================================================
def generate_all_visualizations():
    print("Generating Visualizations for Final Report...\n")

    # ---------------------------------------------------------------------
    # 1. CLASSIFICATION VISUALIZATIONS
    # ---------------------------------------------------------------------
    try:
        class_metrics = pd.read_csv("output_accuracy_classification_model_comparison_metrics.csv")
        class_preds = pd.read_csv("output_accuracy_classification_all_model_predictions.csv")

        # Best Model Comparison
        create_model_comparison_plots(class_metrics, 'accuracy',
                                      "Classification Accuracy Comparison",
                                      "class_model_comparison_accuracy.png")

        # Confusion Matrix for the best model (Logistic Regression from your last run)
        best_class_model = class_metrics.loc[0, 'model']
        best_class_preds = class_preds[class_preds['model'] == best_class_model]
        create_confusion_matrix_plot(best_class_preds, 'Foundation_Type',
                                     f"Confusion Matrix: {best_class_model}",
                                     "class_confusion_matrix.png")

        # Feature Importances
        for model in ['random', 'lightgbm']:
            try:
                imp_df = pd.read_csv(f"output_accuracy_classification_feature_importance_{model}.csv")
                create_feature_importance_plot(imp_df,
                                               f"Feature Importance ({model.title()}) - Classification",
                                               f"class_feature_importance_{model}.png")
            except FileNotFoundError:
                pass

    except FileNotFoundError:
        print("Classification CSVs not found in current directory. Run model.ipynb first.")


    # ---------------------------------------------------------------------
    # 2. PREDICTION VISUALIZATIONS
    # ---------------------------------------------------------------------
    try:
        pred_metrics = pd.read_csv("output_accuracy_prediction_model_comparison_metrics.csv")
        pred_preds = pd.read_csv("output_accuracy_prediction_best_model_predictions.csv")

        # Best Model Comparison
        create_model_comparison_plots(pred_metrics, 'rmse',
                                      "Prediction RMSE Comparison (Lower is Better)",
                                      "pred_model_comparison_rmse.png")

        # Parity & Residual Plots for the best model
        best_pred_model = pred_metrics.loc[0, 'model']
        create_parity_plot(pred_preds, 'Pier total Depth (ft)',
                           f"Parity Plot: {best_pred_model}",
                           "pred_parity_plot.png")

        create_residual_plot(pred_preds, 'Pier total Depth (ft)',
                             f"Residual Plot: {best_pred_model}",
                             "pred_residual_plot.png")

        # Feature Importances
        for model in ['random_forest', 'lightgbm']:
            try:
                imp_df = pd.read_csv(f"output_accuracy_prediction_feature_importance_{model}.csv")
                create_feature_importance_plot(imp_df,
                                               f"Feature Importance ({model.replace('_', ' ').title()}) - Prediction",
                                               f"pred_feature_importance_{model}.png")
            except FileNotFoundError:
                pass

    except FileNotFoundError:
        print("Prediction CSVs not found in current directory. Run model.ipynb first.")

    # ---------------------------------------------------------------------
    # DOWNLOAD FIGURES
    # ---------------------------------------------------------------------
    print("\nInitiating downloads for all generated figures...")
    for file_path in generated_images:
        try:
            files.download(file_path)
        except Exception as e:
             print(f"Could not download {file_path}. Error: {e}")

if __name__ == "__main__":
    generate_all_visualizations()